In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/career_dataset.csv")

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (10000, 46)


,student_id,age,gender,city,branch,tenth_percentage,twelfth_percentage,cgpa,backlogs,attendance_percentage,...,remote_preference,higher_studies_interest,academic_index,skill_index,experience_index,communication_index,employability_index,placed,placement_role,package_lpa
0,10001,23,Male,Hyderabad,Mechanical,64.64,61.62,6.41,3,66.08,...,No,No,63.99,55.14,38,79.08,59.27,1,Business Analyst,11.92
1,10002,24,Female,Jaipur,EEE,82.48,60.66,6.67,0,80.86,...,Yes,No,70.78,48.66,96,68.31,66.72,1,Data Scientist,12.49
2,10003,22,Male,Hyderabad,CSE,78.24,68.75,7.77,0,55.40,...,Yes,No,72.69,59.19,22,47.27,54.26,1,BI Analyst,9.68
3,10004,24,Male,Mumbai,CSE,78.57,70.68,6.06,0,74.77,...,Yes,No,68.34,75.36,52,65.92,67.32,1,Data Analyst,10.15
4,10005,24,Male,Bhopal,Mechanical,77.44,87.71,6.59,0,86.57,...,Yes,No,75.65,56.07,53,70.63,62.72,0,Not Placed,0.00


In [2]:
print("Salary Statistics:")
print(df["package_lpa"].describe())

print("\nMissing Values:")
print(df["package_lpa"].isnull().sum())

Salary Statistics:
count    10000.000000
mean         8.845012
std          3.841096
min          0.000000
25%          8.810000
50%         10.050000
75%         11.060000
max         15.640000
Name: package_lpa, dtype: float64

Missing Values:
0


In [3]:
print("Salary Statistics:")
print(df["package_lpa"].describe())

print("\nMissing Values:")
print(df["package_lpa"].isnull().sum())

Salary Statistics:
count    10000.000000
mean         8.845012
std          3.841096
min          0.000000
25%          8.810000
50%         10.050000
75%         11.060000
max         15.640000
Name: package_lpa, dtype: float64

Missing Values:
0


In [4]:
drop_columns = [
    "student_id",
    "placed",
    "placement_role",
    "package_lpa",
    "preferred_role",
    "preferred_domain",
    "preferred_location",
    "remote_preference",
    "higher_studies_interest"
]

X = df.drop(columns=drop_columns)
y = df["package_lpa"]

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:", numeric_features)
print("Categorical Features:", categorical_features)

Numerical Features: ['age', 'tenth_percentage', 'twelfth_percentage', 'cgpa', 'backlogs', 'attendance_percentage', 'python_score', 'sql_score', 'excel_score', 'powerbi_score', 'statistics_score', 'ml_score', 'java_score', 'web_dev_score', 'data_visualization_score', 'coding_score', 'communication_score', 'aptitude_score', 'logical_reasoning_score', 'presentation_score', 'teamwork_score', 'leadership_score', 'internship_count', 'internship_months', 'work_experience_months', 'industry_projects', 'personal_projects', 'certification_count', 'project_quality_score', 'academic_index', 'skill_index', 'experience_index', 'communication_index', 'employability_index']
Categorical Features: ['gender', 'city', 'branch']


C:\Users\Admin\AppData\Local\Temp\ipykernel_20084\2379350837.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (8000, 37)
Testing Data: (2000, 37)


In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

salary_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

salary_model.fit(X_train, y_train)

print("Salary model trained successfully!")

Salary model trained successfully!


In [9]:
salary_pred = salary_model.predict(X_test)

print("Sample Predictions:")
print(salary_pred[:10])

Sample Predictions:
[11.742       9.0239     10.1489     10.0086      7.65016667  8.8242
 10.43846667  7.2117      7.8336      7.80983333]


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, salary_pred)
rmse = np.sqrt(mean_squared_error(y_test, salary_pred))
r2 = r2_score(y_test, salary_pred)

print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R²  :", round(r2, 3))

MAE : 2.692
RMSE: 3.711
R²  : 0.097


In [11]:
import joblib

joblib.dump(
    salary_model,
    "../models/salary_prediction_pipeline.pkl"
)

print("Salary model saved successfully!")

Salary model saved successfully!
